In [1]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

NETID = "ddemarco"

df = pd.read_csv(
    f"{NETID}_project_summary.csv",
    sep=";",
    dtype={"commit_sha1": str}
)

df["time"] = pd.to_numeric(df["time"])
df["datetime"] = pd.to_datetime(df["time"], unit="s", utc=True)
df["Month"] = df["datetime"].dt.tz_localize(None).dt.to_period("M")

print("Commits:", len(df))
print("Projects:", df["project_wocid"].nunique())
df.head()


Commits: 10481
Projects: 10


,project_wocid,commit_sha1,author,time,commit message,datetime,Month
0,sebwouters_chemps2,002818006570601b737a1c290afd875394c116b1,Sebastian Wouters <sebastianwouters@gmail.com>,1464086967,Faster Wigner routine - deprecate prime factor...,2016-05-24 10:49:27+00:00,2016-05
1,sebwouters_chemps2,00348b233ce6cc54dd18625fe994d25b09ed9b71,Sebastian Wouters <sebastianwouters@gmail.com>,1463134405,Reshuffle 3-RDM internally\n,2016-05-13 10:13:25+00:00,2016-05
2,sebwouters_chemps2,00620282a5d6b1e11c3fd3b4f6013e2a9a12fc08,Toon Verstraelen <Toon.Verstraelen@UGent.be>,1430003605,Minor corrections in PyCheMPS2 installation in...,2015-04-25 23:13:25+00:00,2015-04
3,sebwouters_chemps2,00687e9aa79b1aa8d6b7770cb9c612378934919f,Sebastian Wouters <sebastianwouters@gmail.com>,1423523915,DMRGSCF plugin for psi4 works :-)\n,2015-02-09 23:18:35+00:00,2015-02
4,sebwouters_chemps2,01e5ce2ee76a6bc080c14ef9f48a7f2c4913d3cf,Sebastian Wouters <sebastianwouters@gmail.com>,1464096136,Official version 1.7 release\n,2016-05-24 13:22:16+00:00,2016-05


In [2]:

monthly_data = {}

for project, group in df.groupby("project_wocid"):
    counts = group.groupby("Month").size()

    all_months = pd.period_range(
        start=group["Month"].min(),
        end=group["Month"].max(),
        freq="M"
    )

    counts = counts.reindex(all_months, fill_value=0)

    ts = pd.DataFrame({
        "Month": all_months.astype(str),
        "#Commits": counts.to_numpy()
    })

    monthly_data[project] = ts

    ts.to_csv(
        f"{NETID}_commits_timeseries_{project}.csv",
        sep=";",
        index=False
    )

print("Created", len(monthly_data), "monthly CSV files.")
monthly_data["sebwouters_chemps2"].head(10)


Created 10 monthly CSV files.


,Month,#Commits
0,2013-11,3
1,2013-12,18
2,2014-01,11
3,2014-02,2
4,2014-03,3
5,2014-04,10
6,2014-05,2
7,2014-06,4
8,2014-07,10
9,2014-08,17


In [3]:

for project, ts in monthly_data.items():
    x = pd.to_datetime(ts["Month"], format="%Y-%m")

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(x, ts["#Commits"], linewidth=1.5)
    ax.set_title(project)
    ax.set_xlabel("Time (YYYY-MM)")
    ax.set_ylabel("Number of Commits")
    ax.grid(True, alpha=0.4)

    fig.autofmt_xdate()
    fig.tight_layout()

    fig.savefig(
        f"{NETID}_timeseries_{project}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

print("Saved 10 plots at 300 DPI.")


Saved 10 plots at 300 DPI.


## Interpretation

The 10 projects showed different commit activity patterns, including rising, declining, and irregular trends.

- **aidenlab_juicer:** Irregular activity with 5 gaps lasting at least 3 months. Its longest gap was 11 months.
- **f-fathurrahman_pwdft.jl:** Declining activity with 1 gap lasting at least 3 months. Its longest gap was 3 months.
- **icrar_shark:** Declining activity with 2 gaps lasting at least 3 months. Its longest gap was 6 months.
- **mathlab_pydmd:** Rising activity with 0 gaps lasting at least 3 months. Its longest gap was 2 months.
- **nenadmarkus_wlrn:** Irregular activity with 6 gaps lasting at least 3 months. Its longest gap was 35 months.
- **nuest_o2r-project.github.io:** Declining activity with 2 gaps lasting at least 3 months. Its longest gap was 18 months.
- **qiboteam_qibojit:** Irregular activity with 0 gaps lasting at least 3 months. Its longest gap was 1 month.
- **rednaga_apkid:** Irregular activity with 1 gap lasting at least 3 months. Its longest gap was 3 months.
- **sebwouters_chemps2:** Declining activity with 5 gaps lasting at least 3 months. Its longest gap was 37 months.
- **yixuan_spectra:** Irregular activity with 8 gaps lasting at least 3 months. Its longest gap was 5 months.

Overall, most projects showed irregular or declining activity rather than steady development. `mathlab_pydmd` showed an increasing trend, while `sebwouters_chemps2` and `nenadmarkus_wlrn` showed long inactivity times. The plots had no clear, consistent seasonal patterns across the projects.

In [4]:

def find_gaps(ts):
    gaps = []
    start = None

    for i, count in enumerate(ts["#Commits"]):
        if count == 0 and start is None:
            start = i

        if count > 0 and start is not None:
            gaps.append((start, i - 1))
            start = None

    if start is not None:
        gaps.append((start, len(ts) - 1))

    return [
        {
            "start": ts.iloc[a]["Month"],
            "end": ts.iloc[b]["Month"],
            "length": b - a + 1
        }
        for a, b in gaps
    ]


gap_rows = []

for project, ts in monthly_data.items():
    gaps = find_gaps(ts)
    longest = max(
        gaps,
        key=lambda g: g["length"],
        default=None
    )

    if longest:
        end_period = pd.Period(longest["end"], freq="M")
        after = ts[
            pd.PeriodIndex(ts["Month"], freq="M") > end_period
        ]
        commits_after = int(after["#Commits"].sum())
    else:
        commits_after = 0

    gap_rows.append({
        "project": project,
        "LongestGapStart": longest["start"] if longest else "",
        "LongestGapEnd": longest["end"] if longest else "",
        "LongestGapLength": longest["length"] if longest else 0,
        "NumCommitsAfterLastGap": commits_after,
        "NumberOfGapsInTimeline": sum(
            g["length"] >= 3 for g in gaps
        )
    })

gap_stats = pd.DataFrame(gap_rows)
gap_stats


,project,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline
0,aidenlab_juicer,2023-03,2024-01,11,30,5
1,f-fathurrahman_pwdft.jl,2025-05,2025-07,3,12,1
2,icrar_shark,2021-04,2021-09,6,294,2
3,mathlab_pydmd,2019-01,2019-02,2,940,0
4,nenadmarkus_wlrn,2019-06,2022-04,35,217,6
5,nuest_o2r-project.github.io,2023-10,2025-03,18,3,2
6,qiboteam_qibojit,2025-07,2025-07,1,100,0
7,rednaga_apkid,2021-09,2021-11,3,432,1
8,sebwouters_chemps2,2022-04,2025-04,37,8,5
9,yixuan_spectra,2017-07,2017-11,5,748,8


In [5]:

base_stats = (
    df.groupby("project_wocid")
    .agg(
        ncommits=("commit_sha1", "nunique"),
        nauthors=("author", "nunique"),
        **{
            "from": ("datetime", "min"),
            "to": ("datetime", "max")
        }
    )
    .reset_index()
    .rename(columns={"project_wocid": "Project"})
)

for col in ["from", "to"]:
    base_stats[col] = base_stats[col].dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

base_stats.head()


,Project,ncommits,nauthors,from,to
0,aidenlab_juicer,510,44,2015-12-24 18:59:23,2025-09-24 16:22:25
1,f-fathurrahman_pwdft.jl,2112,7,2018-02-19 03:45:59,2025-08-21 11:09:22
2,icrar_shark,1595,30,2017-04-20 04:44:19,2025-07-08 04:15:11
3,mathlab_pydmd,1130,52,2017-06-12 12:33:47,2023-11-16 18:43:11
4,nenadmarkus_wlrn,322,10,2016-02-15 16:50:39,2025-07-20 13:49:52


In [6]:

github_data = pd.DataFrame([
    ["sebwouters_chemps2", 73, 38, "2025-07-15"],
    ["f-fathurrahman_pwdft.jl", 122, 23, "2026-09-17"],
    ["nenadmarkus_wlrn", 28, 8, "2025-07-20"],
    ["aidenlab_juicer", 490, 192, "2025-08-31"],
    ["qiboteam_qibojit", 18, 4, "2026-09-01"],
    ["nuest_o2r-project.github.io", 1, 2, "2025-04-23"],
    ["rednaga_apkid", 2578, 344, "2026-09-02"],
    ["icrar_shark", 33, 16, "2026-09-09"],
    ["yixuan_spectra", 866, 139, "2026-07-04"],
    ["mathlab_pydmd", 128, 11, "2025-02-04"]
], columns=[
    "Project", "nstars", "nforks", "lastGHCommitDate"
])

stats = (
    base_stats
    .merge(github_data, on="Project", validate="one_to_one")
    .merge(
        gap_stats.rename(columns={"project": "Project"}),
        on="Project",
        validate="one_to_one"
    )
)

stats["ActivityPattern"] = ""

stats.to_csv(
    f"{NETID}_project_stats.csv",
    sep=";",
    index=False
)

stats


,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,ActivityPattern
0,aidenlab_juicer,510,44,2015-12-24 18:59:23,2025-09-24 16:22:25,490,192,2025-08-31,2023-03,2024-01,11,30,5,
1,f-fathurrahman_pwdft.jl,2112,7,2018-02-19 03:45:59,2025-08-21 11:09:22,122,23,2026-09-17,2025-05,2025-07,3,12,1,
2,icrar_shark,1595,30,2017-04-20 04:44:19,2025-07-08 04:15:11,33,16,2026-09-09,2021-04,2021-09,6,294,2,
3,mathlab_pydmd,1130,52,2017-06-12 12:33:47,2023-11-16 18:43:11,128,11,2025-02-04,2019-01,2019-02,2,940,0,
4,nenadmarkus_wlrn,322,10,2016-02-15 16:50:39,2025-07-20 13:49:52,28,8,2025-07-20,2019-06,2022-04,35,217,6,
5,nuest_o2r-project.github.io,670,30,2016-01-19 16:18:32,2025-04-23 11:08:15,1,2,2025-04-23,2023-10,2025-03,18,3,2,
6,qiboteam_qibojit,1304,26,2021-04-16 11:23:47,2025-11-06 06:07:04,18,4,2026-09-01,2025-07,2025-07,1,100,0,
7,rednaga_apkid,1068,76,2016-06-25 01:07:26,2025-11-02 17:17:03,2578,344,2026-09-02,2021-09,2021-11,3,432,1,
8,sebwouters_chemps2,771,10,2013-11-18 15:36:41,2025-07-15 18:53:34,73,38,2025-07-15,2022-04,2025-04,37,8,5,
9,yixuan_spectra,999,42,2015-09-10 03:54:19,2025-08-26 09:33:36,866,139,2026-07-04,2017-07,2017-11,5,748,8,


In [17]:
activity_patterns = {
    "aidenlab_juicer": "irregular",
    "f-fathurrahman_pwdft.jl": "declining",
    "icrar_shark": "declining",
    "mathlab_pydmd": "rising",
    "nenadmarkus_wlrn": "irregular",
    "nuest_o2r-project.github.io": "declining",
    "qiboteam_qibojit": "irregular",
    "rednaga_apkid": "irregular",
    "sebwouters_chemps2": "declining",
    "yixuan_spectra": "irregular"
}

stats["ActivityPattern"] = stats["Project"].map(activity_patterns)

print(stats[["Project", "ActivityPattern"]].to_string(index=False))

                    Project ActivityPattern
            aidenlab_juicer       irregular
    f-fathurrahman_pwdft.jl       declining
                icrar_shark       declining
              mathlab_pydmd          rising
           nenadmarkus_wlrn       irregular
nuest_o2r-project.github.io       declining
           qiboteam_qibojit       irregular
              rednaga_apkid       irregular
         sebwouters_chemps2       declining
             yixuan_spectra       irregular


In [7]:

gap_commit_rows = []

for _, row in gap_stats.iterrows():
    project = row["project"]

    project_commits = (
        df[df["project_wocid"] == project]
        .sort_values("time")
    )

    if not row["LongestGapStart"]:
        continue

    gap_start = pd.Period(row["LongestGapStart"], freq="M")
    gap_end = pd.Period(row["LongestGapEnd"], freq="M")

    before = project_commits[
        project_commits["Month"] < gap_start
    ].tail(10)

    after = project_commits[
        project_commits["Month"] > gap_end
    ].head(10)

    for period_label, selected in [
        ("pre", before),
        ("post", after)
    ]:
        for _, commit in selected.iterrows():
            gap_commit_rows.append({
                "pre/post": period_label,
                "prj": project,
                "commit": commit["commit_sha1"],
                "author": commit["author"],
                "time": commit["time"],
                "message": commit["commit message"]
            })

gap_commits = pd.DataFrame(
    gap_commit_rows,
    columns=["pre/post", "prj", "commit", "author", "time", "message"]
)

gap_commits.to_csv(
    f"{NETID}_project_gap_commits.csv",
    sep=";",
    index=False
)

print("Gap commit records:", len(gap_commits))
print(gap_commits.groupby(["prj", "pre/post"]).size())


Gap commit records: 191
prj                          pre/post
aidenlab_juicer              post        10
                             pre         10
f-fathurrahman_pwdft.jl      post        10
                             pre         10
icrar_shark                  post        10
                             pre         10
mathlab_pydmd                post        10
                             pre         10
nenadmarkus_wlrn             post        10
                             pre         10
nuest_o2r-project.github.io  post         3
                             pre         10
qiboteam_qibojit             post        10
                             pre         10
rednaga_apkid                post        10
                             pre         10
sebwouters_chemps2           post         8
                             pre         10
yixuan_spectra               post        10
                             pre         10
dtype: int64


In [8]:

def classify_theme(message, author=""):
    text = str(message).lower()
    author = str(author).lower()

    if "dependabot" in author or "renovate[bot]" in author:
        return "Automated bot contributions"

    if any(x in text for x in [
        "security", "vulnerability", "cve-", "exploit"
    ]):
        return "Security patches"

    if any(x in text for x in [
        "dependabot", "renovate", "bump ", "upgrade dependency",
        "update dependency", "requirements.txt"
    ]):
        return "Dependency updates"

    if any(x in text for x in [
        "release", "changelog", "version bump", "tag v"
    ]):
        return "Release"

    if any(x in text for x in [
        "readme", "documentation", "docs", "docstring", "tutorial"
    ]):
        return "Documentation updates"

    if any(x in text for x in [
        "fix", "bug", "error", "crash", "correct", "patch"
    ]):
        return "Bug fixes"

    if any(x in text for x in [
        "add", "implement", "feature", "support", "introduce",
        "new ", "enable", "create"
    ]):
        return "Feature development"

    return "Other"


gap_commits["SuggestedTheme"] = gap_commits.apply(
    lambda r: classify_theme(r["message"], r["author"]),
    axis=1
)

gap_commits[[
    "prj", "pre/post", "message", "SuggestedTheme"
]].head(20)


,prj,pre/post,message,SuggestedTheme
0,aidenlab_juicer,pre,Update juicer.sh\n\nI had to remove ${!SLURM_J...,Other
1,aidenlab_juicer,pre,Update juicer_postprocessing.sh\n\nExpose --th...,Other
2,aidenlab_juicer,pre,Update juicer_postprocessing.sh\n\nmove --thre...,Other
3,aidenlab_juicer,pre,Merge pull request #282 from adadiehl/patch-1\...,Bug fixes
4,aidenlab_juicer,pre,Provide default value to --threads\n,Other
5,aidenlab_juicer,pre,Merge pull request #286 from adadiehl/main\n\n...,Other
6,aidenlab_juicer,pre,"Update README\n\nAdd more links to ENCODE, mak...",Documentation updates
7,aidenlab_juicer,pre,Merge pull request #304 from aidenlab/nchernia...,Documentation updates
8,aidenlab_juicer,pre,Merge a12df405a0e16eaee432f3f2980bcecab91a0240...,Feature development
9,aidenlab_juicer,pre,Merge f34dcc4eece4c6d458dbe3a8e4c8cce4846446a0...,Feature development


In [9]:

for project in sorted(gap_commits["prj"].unique()):
    print("\n" + "=" * 75)
    print("PROJECT:", project)

    for period in ["pre", "post"]:
        print("\n", period.upper(), "GAP COMMITS")

        selected = gap_commits[
            (gap_commits["prj"] == project) &
            (gap_commits["pre/post"] == period)
        ]

        for _, r in selected.iterrows():
            print(
                f"{r['SuggestedTheme']} | "
                f"{str(r['message']).strip()[:180]}"
            )



PROJECT: aidenlab_juicer

 PRE GAP COMMITS
Other | Update juicer.sh

I had to remove ${!SLURM_JOB_ID} and replace with ${SLURM_JOB_ID} on Debian11?
Other | Update juicer_postprocessing.sh

Expose --threads option for Arrowhead
Other | Update juicer_postprocessing.sh

move --threads flag to be directly after arrowhead command
Bug fixes | Merge pull request #282 from adadiehl/patch-1

Update juicer_postprocessing.sh
Other | Provide default value to --threads
Other | Merge pull request #286 from adadiehl/main

Provide default value to --threads
Documentation updates | Update README

Add more links to ENCODE, make clear certain systems are no longer under active maintenance / development.
Documentation updates | Merge pull request #304 from aidenlab/nchernia-patch-1

Update README
Feature development | Merge a12df405a0e16eaee432f3f2980bcecab91a0240 into 290e443dd06f737109e5d93b4f4ae3cadd94ddef
Feature development | Merge f34dcc4eece4c6d458dbe3a8e4c8cce4846446a0 into 290e443dd06f737109e5d9

In [10]:

def top_two_themes(messages, authors):
    themes = [
        classify_theme(m, a)
        for m, a in zip(messages, authors)
    ]

    if not themes:
        return ""

    counts = pd.Series(themes).value_counts()
    return ":".join(counts.index[:2])


theme_rows = []

for project in sorted(df["project_wocid"].unique()):
    subset = gap_commits[gap_commits["prj"] == project]

    before = subset[subset["pre/post"] == "pre"]
    after = subset[subset["pre/post"] == "post"]

    theme_rows.append({
        "Project": project,
        "BeforeThemes": top_two_themes(
            before["message"], before["author"]
        ),
        "AfterThemes": top_two_themes(
            after["message"], after["author"]
        )
    })

theme_stats = pd.DataFrame(theme_rows)
theme_stats


,Project,BeforeThemes,AfterThemes
0,aidenlab_juicer,Other:Documentation updates,Feature development:Other
1,f-fathurrahman_pwdft.jl,Other,Other:Bug fixes
2,icrar_shark,Other:Bug fixes,Other:Feature development
3,mathlab_pydmd,Feature development:Bug fixes,Feature development:Documentation updates
4,nenadmarkus_wlrn,Other:Feature development,Other:Feature development
5,nuest_o2r-project.github.io,Other:Feature development,Feature development:Other
6,qiboteam_qibojit,Other:Security patches,Other:Bug fixes
7,rednaga_apkid,Other:Feature development,Other
8,sebwouters_chemps2,Bug fixes:Other,Feature development:Dependency updates
9,yixuan_spectra,Other:Feature development,Other:Feature development


In [11]:

contributor_rows = []

for project in sorted(df["project_wocid"].unique()):
    subset = gap_commits[gap_commits["prj"] == project]

    pre = set(
        subset[subset["pre/post"] == "pre"]["author"]
    )
    post = set(
        subset[subset["pre/post"] == "post"]["author"]
    )

    contributor_rows.append({
        "Project": project,
        "BeforeAuthors": len(pre),
        "AfterAuthors": len(post),
        "ReturningAuthors": len(pre & post),
        "NewAfterGapAuthors": len(post - pre),
        "NewAuthorNames": "; ".join(sorted(post - pre))
    })

contributor_comparison = pd.DataFrame(contributor_rows)
contributor_comparison


,Project,BeforeAuthors,AfterAuthors,ReturningAuthors,NewAfterGapAuthors,NewAuthorNames
0,aidenlab_juicer,5,5,0,5,David Weisz <weisz@bcm.edu>; Emerson Jacobson ...
1,f-fathurrahman_pwdft.jl,1,1,1,0,
2,icrar_shark,3,2,2,0,
3,mathlab_pydmd,3,2,0,2,Unknown <loiseau.jc@gmail.com>; Vahid S. Bokha...
4,nenadmarkus_wlrn,4,1,0,1,nenad <nenad.markus@protonmail.com>
5,nuest_o2r-project.github.io,2,2,0,2,Daniel Nüst <daniel.nuest@tu-dresden.de>; Dani...
6,qiboteam_qibojit,4,2,1,1,pre-commit-ci[bot] <66853113+pre-commit-ci[bot...
7,rednaga_apkid,4,1,1,0,
8,sebwouters_chemps2,1,3,1,2,Cristian Le <git@lecris.dev>; Eisuke Kawashima...
9,yixuan_spectra,2,4,1,3,Angelos Mantzaflaris <Angelos.Mantzaflaris@oea...


In [12]:

recent_rows = []

for project, group in df.groupby("project_wocid"):
    latest_10 = group.sort_values("time").tail(10)

    recent_rows.append({
        "Project": project,
        "RecentThemes": top_two_themes(
            latest_10["commit message"],
            latest_10["author"]
        )
    })

recent_stats = pd.DataFrame(recent_rows)

stats = (
    stats
    .merge(theme_stats, on="Project", validate="one_to_one")
    .merge(recent_stats, on="Project", validate="one_to_one")
)

cutoff = pd.Timestamp("2025-04-01", tz="UTC")

stats["Currentstatus"] = (
    pd.to_datetime(stats["lastGHCommitDate"], utc=True)
    .apply(lambda date: "Inactive" if date < cutoff else "Active")
)

stats[["Project", "Currentstatus", "RecentThemes"]]


,Project,Currentstatus,RecentThemes
0,aidenlab_juicer,Active,Other:Documentation updates
1,f-fathurrahman_pwdft.jl,Active,Other:Bug fixes
2,icrar_shark,Active,Feature development:Other
3,mathlab_pydmd,Inactive,Other:Bug fixes
4,nenadmarkus_wlrn,Active,Other:Feature development
5,nuest_o2r-project.github.io,Active,Other:Feature development
6,qiboteam_qibojit,Active,Other:Bug fixes
7,rednaga_apkid,Active,Other:Feature development
8,sebwouters_chemps2,Active,Other:Feature development
9,yixuan_spectra,Active,Feature development:Other


In [13]:

for col in [
    "HypothesizedGapReason",
    "HasRecovered",
    "WhyRecovered",
    "WhoRecovered",
    "Notes"
]:
    stats[col] = ""

for i, row in stats.iterrows():
    project = row["Project"]

    if row["LongestGapLength"] == 0:
        stats.at[i, "HypothesizedGapReason"] = "N/A active Project"
        stats.at[i, "HasRecovered"] = "N/A"
    else:
        stats.at[i, "HasRecovered"] = (
            "Yes" if row["NumCommitsAfterLastGap"] > 0 else "No"
        )

stats.head()


,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,...,ActivityPattern,BeforeThemes,AfterThemes,RecentThemes,Currentstatus,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered,Notes
0,aidenlab_juicer,510,44,2015-12-24 18:59:23,2025-09-24 16:22:25,490,192,2025-08-31,2023-03,2024-01,...,,Other:Documentation updates,Feature development:Other,Other:Documentation updates,Active,,Yes,,,
1,f-fathurrahman_pwdft.jl,2112,7,2018-02-19 03:45:59,2025-08-21 11:09:22,122,23,2026-09-17,2025-05,2025-07,...,,Other,Other:Bug fixes,Other:Bug fixes,Active,,Yes,,,
2,icrar_shark,1595,30,2017-04-20 04:44:19,2025-07-08 04:15:11,33,16,2026-09-09,2021-04,2021-09,...,,Other:Bug fixes,Other:Feature development,Feature development:Other,Active,,Yes,,,
3,mathlab_pydmd,1130,52,2017-06-12 12:33:47,2023-11-16 18:43:11,128,11,2025-02-04,2019-01,2019-02,...,,Feature development:Bug fixes,Feature development:Documentation updates,Other:Bug fixes,Inactive,,Yes,,,
4,nenadmarkus_wlrn,322,10,2016-02-15 16:50:39,2025-07-20 13:49:52,28,8,2025-07-20,2019-06,2022-04,...,,Other:Feature development,Other:Feature development,Other:Feature development,Active,,Yes,,,


In [22]:
corrected_themes = {
    "aidenlab_juicer": (
        "Bug fixes:Documentation updates",
        "Feature development:Bug fixes"
    ),
    "f-fathurrahman_pwdft.jl": (
        "Feature development:Other",
        "Feature development:Bug fixes"
    ),
    "icrar_shark": (
        "Feature development:Bug fixes",
        "Feature development:Bug fixes"
    ),
    "mathlab_pydmd": (
        "Bug fixes:Documentation updates",
        "Feature development:Documentation updates"
    ),
    "nenadmarkus_wlrn": (
        "Feature development:Dependency updates",
        "Feature development:Other"
    ),
    "nuest_o2r-project.github.io": (
        "Documentation updates:Other",
        "Documentation updates:Feature development"
    ),
    "qiboteam_qibojit": (
        "Dependency updates:Feature development",
        "Feature development:Automated bot contributions"
    ),
    "rednaga_apkid": (
        "Feature development:Other",
        "Security patches:Other"
    ),
    "sebwouters_chemps2": (
        "Bug fixes:Dependency updates",
        "Bug fixes:Dependency updates"
    ),
    "yixuan_spectra": (
        "Documentation updates:Other",
        "Bug fixes:Feature development"
    )
}

stats["BeforeThemes"] = stats["Project"].map(
    lambda p: corrected_themes[p][0]
)

stats["AfterThemes"] = stats["Project"].map(
    lambda p: corrected_themes[p][1]
)

print(
    stats[["Project", "BeforeThemes", "AfterThemes"]]
    .to_string(index=False)
)

                    Project                           BeforeThemes                                     AfterThemes
            aidenlab_juicer        Bug fixes:Documentation updates                   Feature development:Bug fixes
    f-fathurrahman_pwdft.jl              Feature development:Other                   Feature development:Bug fixes
                icrar_shark          Feature development:Bug fixes                   Feature development:Bug fixes
              mathlab_pydmd        Bug fixes:Documentation updates       Feature development:Documentation updates
           nenadmarkus_wlrn Feature development:Dependency updates                       Feature development:Other
nuest_o2r-project.github.io            Documentation updates:Other       Documentation updates:Feature development
           qiboteam_qibojit Dependency updates:Feature development Feature development:Automated bot contributions
              rednaga_apkid              Feature development:Other              

In [23]:
gap_explanations = {
    "aidenlab_juicer": (
        "Possible maintenance slowdown or shift in development priorities; README noted some systems were no longer actively maintained.",
        "Yes",
        "Development resumed with Docker support, bwa-mem2 integration, and bug fixes.",
        "Different contributors in the sampled commits, including David Weisz and Emerson Jacobson.",
        "The project had an 11-month inactivity gap and 5 extended gaps. Afterward, contributors added containerization and alignment improvements. The sampled post-gap authors differ from those immediately before the gap."
    ),
    "f-fathurrahman_pwdft.jl": (
        "Reason unclear; possibly a temporary pause in individual development.",
        "Yes",
        "The developer resumed solver implementation and bug fixes.",
        "Fadjar Fathurrahman, the same contributor as before the gap.",
        "The project experienced a 3-month inactivity gap. Development resumed with fixes and new numerical solver functionality. The same author contributed before and after the gap."
    ),
    "icrar_shark": (
        "Reason unclear; possibly a pause between research and development phases.",
        "Yes",
        "Development resumed with galaxy-model changes and new analysis scripts.",
        "Claudia Lagos and Rodrigo Tobar returned in the sampled commits.",
        "The project had a 6-month inactivity gap. Subsequent commits introduced research analysis scripts and model improvements. Two contributors appeared on both sides of the gap."
    ),
    "mathlab_pydmd": (
        "Possibly a short transition between development contributions; no confirmed cause.",
        "Yes",
        "New contributors implemented optimal DMD functionality and tutorials.",
        "Vahid S. Bokharaie and a contributor using loiseau.jc@gmail.com appeared after the gap.",
        "The longest gap lasted only 2 months, with no gaps of 3 months or more. Development resumed with new DMD implementations and tutorials. The sampled contributors changed across the gap."
    ),
    "nenadmarkus_wlrn": (
        "Possibly a prolonged pause before renewed work on PyTorch and dataset support.",
        "Yes",
        "Development resumed with older code, KITTI dataset loading, and cleanup.",
        "Nenad Markus appears to have returned under a different email address.",
        "The project had a 35-month inactivity gap and 6 extended gaps. Work resumed with dataset loaders and code cleanup. Different author emails appear to represent the same developer."
    ),
    "nuest_o2r-project.github.io": (
        "Possibly reduced project activity or fewer website updates; exact cause unconfirmed.",
        "Yes",
        "The website received navigation improvements and restored project content.",
        "Daniel Nüst contributed the sampled post-gap commits.",
        "The website had an 18-month inactivity gap. Only 3 commits followed, focusing on website content and navigation. This indicates limited resumed activity rather than sustained recovery."
    ),
    "qiboteam_qibojit": (
        "Brief development pause; no evidence of a major interruption.",
        "Yes",
        "Work resumed on GPU-related refactoring, tests, and dependency updates.",
        "Renato Mello returned, alongside automated pre-commit contributions.",
        "The longest gap lasted only 1 month, with no extended gaps. Development continued with GPU refactoring and maintenance. Both human and automated contributions appeared afterward."
    ),
    "rednaga_apkid": (
        "Possibly a temporary pause between detection-rule updates.",
        "Yes",
        "Development resumed with updates to YARA detection rules.",
        "Apkunpacker contributed all 10 sampled post-gap commits.",
        "The project had a 3-month inactivity gap. The first 10 subsequent commits updated YARA detection rules. This suggests continued security-tool maintenance, although the precise reason for the gap is unknown."
    ),
    "sebwouters_chemps2": (
        "Possibly reduced maintenance after a stable release; exact cause unconfirmed.",
        "Yes",
        "Development resumed to address CMake and MPI build compatibility.",
        "Sebastian Wouters returned, joined by Cristian Le and Eisuke Kawashima.",
        "The project experienced the longest gap at 37 months. Only 8 commits followed, mostly addressing build compatibility. Activity resumed but remained limited."
    ),
    "yixuan_spectra": (
        "Possibly a pause following documentation and website maintenance.",
        "Yes",
        "Development resumed with compilation fixes, performance changes, and documentation.",
        "Yixuan Qiu returned, alongside additional contributors.",
        "The project had a 5-month longest gap and 8 extended gaps. Subsequent commits fixed compilation problems and improved performance. Both returning and additional contributors participated."
    )
}

columns = [
    "HypothesizedGapReason",
    "HasRecovered",
    "WhyRecovered",
    "WhoRecovered",
    "Notes"
]

for col_index, col_name in enumerate(columns):
    stats[col_name] = stats["Project"].map(
        lambda p: gap_explanations[p][col_index]
    )

print(stats[["Project", "HasRecovered", "WhoRecovered"]].to_string(index=False))

                    Project HasRecovered                                                                               WhoRecovered
            aidenlab_juicer          Yes Different contributors in the sampled commits, including David Weisz and Emerson Jacobson.
    f-fathurrahman_pwdft.jl          Yes                               Fadjar Fathurrahman, the same contributor as before the gap.
                icrar_shark          Yes                           Claudia Lagos and Rodrigo Tobar returned in the sampled commits.
              mathlab_pydmd          Yes    Vahid S. Bokharaie and a contributor using loiseau.jc@gmail.com appeared after the gap.
           nenadmarkus_wlrn          Yes                     Nenad Markus appears to have returned under a different email address.
nuest_o2r-project.github.io          Yes                                      Daniel Nüst contributed the sampled post-gap commits.
           qiboteam_qibojit          Yes                       Renato Mello 

In [24]:
from pathlib import Path

for _, row in stats.iterrows():
    project = row["Project"]

    reflection = f"""# Project Reflection: {project}

## Commit Activity
This project had {row['ncommits']} commits from {row['nauthors']} authors in the analyzed dataset. Its overall activity pattern was {row['ActivityPattern']}.

## Inactivity Gap
The longest inactivity gap lasted {row['LongestGapLength']} months, from {row['LongestGapStart']} to {row['LongestGapEnd']}. The project had {row['NumberOfGapsInTimeline']} gaps lasting at least three months.

## Development Before and After the Gap
The main themes before the gap were {row['BeforeThemes']}. After the gap, the main themes were {row['AfterThemes']}.

## Possible Reason for Inactivity
{row['HypothesizedGapReason']}

## Recovery
Activity resumed after the gap: {row['HasRecovered']}.

{row['WhyRecovered']}

## Contributors
{row['WhoRecovered']}

## Current Status
The project is classified as {row['Currentstatus']} using the assignment's activity cutoff. Recent commit themes were {row['RecentThemes']}.

## Conclusion
{row['Notes']}
"""

    filename = f"ddemarco_reflection_{project}.md"
    Path(filename).write_text(reflection, encoding="utf-8")

print("Created 10 completed reflection files.")

Created 10 completed reflection files.


In [25]:
required_columns = [
    "Project", "ncommits", "nauthors", "from", "to",
    "nstars", "nforks", "lastGHCommitDate",
    "LongestGapStart", "LongestGapEnd", "LongestGapLength",
    "NumCommitsAfterLastGap", "ActivityPattern",
    "NumberOfGapsInTimeline", "BeforeThemes",
    "AfterThemes", "HypothesizedGapReason",
    "HasRecovered", "WhyRecovered", "WhoRecovered",
    "Currentstatus", "RecentThemes", "Notes"
]

stats = stats[required_columns]

stats.to_csv(
    "ddemarco_project_stats.csv",
    sep=";",
    index=False
)

print("Saved ddemarco_project_stats.csv")
print("Projects:", len(stats))
print("Columns:", len(stats.columns))

Saved ddemarco_project_stats.csv
Projects: 10
Columns: 23


In [26]:
from pathlib import Path

required_files = [
    "ddemarco.ipynb",
    "ddemarco_vis.ipynb",
    "ddemarco_project_summary.csv",
    "ddemarco_project_stats.csv",
    "ddemarco_project_gap_commits.csv"
]

required_files += [
    f"ddemarco_timeseries_{p}.png"
    for p in stats["Project"]
]

required_files += [
    f"ddemarco_reflection_{p}.md"
    for p in stats["Project"]
]

for filename in required_files:
    status = "OK" if Path(filename).exists() else "MISSING"
    print(f"{status}: {filename}")

OK: ddemarco.ipynb
OK: ddemarco_vis.ipynb
OK: ddemarco_project_summary.csv
OK: ddemarco_project_stats.csv
OK: ddemarco_project_gap_commits.csv
OK: ddemarco_timeseries_aidenlab_juicer.png
OK: ddemarco_timeseries_f-fathurrahman_pwdft.jl.png
OK: ddemarco_timeseries_icrar_shark.png
OK: ddemarco_timeseries_mathlab_pydmd.png
OK: ddemarco_timeseries_nenadmarkus_wlrn.png
OK: ddemarco_timeseries_nuest_o2r-project.github.io.png
OK: ddemarco_timeseries_qiboteam_qibojit.png
OK: ddemarco_timeseries_rednaga_apkid.png
OK: ddemarco_timeseries_sebwouters_chemps2.png
OK: ddemarco_timeseries_yixuan_spectra.png
OK: ddemarco_reflection_aidenlab_juicer.md
OK: ddemarco_reflection_f-fathurrahman_pwdft.jl.md
OK: ddemarco_reflection_icrar_shark.md
OK: ddemarco_reflection_mathlab_pydmd.md
OK: ddemarco_reflection_nenadmarkus_wlrn.md
OK: ddemarco_reflection_nuest_o2r-project.github.io.md
OK: ddemarco_reflection_qiboteam_qibojit.md
OK: ddemarco_reflection_rednaga_apkid.md
OK: ddemarco_reflection_sebwouters_chemps2.

In [18]:
stats["ActivityPattern"] = stats["Project"].map(activity_patterns)

print(stats[["Project", "ActivityPattern"]].to_string(index=False))

                    Project ActivityPattern
            aidenlab_juicer       irregular
    f-fathurrahman_pwdft.jl       declining
                icrar_shark       declining
              mathlab_pydmd          rising
           nenadmarkus_wlrn       irregular
nuest_o2r-project.github.io       declining
           qiboteam_qibojit       irregular
              rednaga_apkid       irregular
         sebwouters_chemps2       declining
             yixuan_spectra       irregular


In [19]:

for _, row in stats.iterrows():
    project = row["Project"]

    contributors = contributor_comparison[
        contributor_comparison["Project"] == project
    ].iloc[0]

    content = f"""# Reflection: {project}

## Activity Pattern
Activity classification: TO REVIEW

Total commits: {row['ncommits']}
Number of gaps lasting at least 3 months: {row['NumberOfGapsInTimeline']}

## Longest Inactivity Gap
Start: {row['LongestGapStart']}
End: {row['LongestGapEnd']}
Length: {row['LongestGapLength']} months
Commits after gap: {row['NumCommitsAfterLastGap']}

## Commit Themes
Before gap (suggested): {row['BeforeThemes']}
After gap (suggested): {row['AfterThemes']}
Recent commits (suggested): {row['RecentThemes']}

## Contributors
Returning contributors in sampled commits: {contributors['ReturningAuthors']}
Contributors appearing only in post-gap sample: {contributors['NewAfterGapAuthors']}

## Interpretation
Explain the overall activity pattern shown in the plot.

## Hypothesized Gap Reason
Explain what the commit messages and GitHub evidence suggest.
If the evidence is insufficient, state that the reason is unknown.

## Recovery
Did activity resume? What work was performed?
Were returning or different contributors involved?

## Evidence
Include relevant commit SHA-1s, GitHub issues, or pull requests
supporting your interpretation.
"""

    Path(f"{NETID}_reflection_{project}.md").write_text(
        content,
        encoding="utf-8"
    )

print("Created 10 reflection templates.")


Created 10 reflection templates.


In [20]:

# Replace these example placeholders with evidence-based answers.
# Valid ActivityPattern values:
# steady, rising, declining, U-shaped, cyclical, irregular

# Example of how to edit one project's values:
#
# mask = stats["Project"] == "sebwouters_chemps2"
# stats.loc[mask, "ActivityPattern"] = "irregular"
# stats.loc[mask, "HypothesizedGapReason"] = "Reason unknown"
# stats.loc[mask, "WhyRecovered"] = "Evidence from commits..."
# stats.loc[mask, "WhoRecovered"] = "Returning contributors..."
# stats.loc[mask, "Notes"] = "2-4 sentence reflection..."

stats.to_csv(
    f"{NETID}_project_stats.csv",
    sep=";",
    index=False
)

required_fields = [
    "ActivityPattern",
    "BeforeThemes",
    "AfterThemes",
    "HypothesizedGapReason",
    "HasRecovered",
    "WhyRecovered",
    "WhoRecovered",
    "Currentstatus",
    "RecentThemes",
    "Notes"
]

print("Projects:", len(stats))
print("Missing fields:")
print(
    stats[required_fields]
    .replace("", pd.NA)
    .isna()
    .sum()
)


Projects: 10
Missing fields:
ActivityPattern           0
BeforeThemes              0
AfterThemes               0
HypothesizedGapReason    10
HasRecovered              0
WhyRecovered             10
WhoRecovered             10
Currentstatus             0
RecentThemes              0
Notes                    10
dtype: int64


In [21]:
with open("ddemarco_gap_review.txt", "w", encoding="utf-8") as f:
    for project in sorted(gap_commits["prj"].unique()):
        f.write(f"\n{'='*60}\nPROJECT: {project}\n")

        for period in ["pre", "post"]:
            f.write(f"\n--- {period.upper()} GAP COMMITS ---\n")

            subset = gap_commits[
                (gap_commits["prj"] == project) &
                (gap_commits["pre/post"] == period)
            ]

            for _, row in subset.iterrows():
                f.write(
                    f"\nAuthor: {row['author']}\n"
                    f"Message: {row['message']}\n"
                )

print("Created ddemarco_gap_review.txt")

Created ddemarco_gap_review.txt


In [2]:
import pandas as pd
from pathlib import Path

# Check the 10 monthly CSVs
stats = pd.read_csv("ddemarco_project_stats.csv", sep=";")

for project in stats["Project"]:
    path = Path(f"ddemarco_commits_timeseries_{project}.csv")
    df = pd.read_csv(path, sep=";")

    headers_ok = list(df.columns) == ["Month", "#Commits"]
    no_missing = not df.isna().any().any()

    print(project, "Headers:", headers_ok, "Missing:", not no_missing)

# Check final stats
final = pd.read_csv("ddemarco_project_stats.csv", sep=";")

print("\nProjects:", len(final))
print("Columns:", len(final.columns))
print("Missing values:", final.isna().sum().sum())

# Check Interpretation exists in notebook
import json

with open("ddemarco_vis.ipynb", encoding="utf-8") as f:
    notebook = json.load(f)

interpretation_found = any(
    cell["cell_type"] == "markdown"
    and "interpretation" in "".join(cell["source"]).lower()
    for cell in notebook["cells"]
)

print("Interpretation found:", interpretation_found)

aidenlab_juicer Headers: True Missing: False
f-fathurrahman_pwdft.jl Headers: True Missing: False
icrar_shark Headers: True Missing: False
mathlab_pydmd Headers: True Missing: False
nenadmarkus_wlrn Headers: True Missing: False
nuest_o2r-project.github.io Headers: True Missing: False
qiboteam_qibojit Headers: True Missing: False
rednaga_apkid Headers: True Missing: False
sebwouters_chemps2 Headers: True Missing: False
yixuan_spectra Headers: True Missing: False

Projects: 10
Columns: 23
Missing values: 0
Interpretation found: False
